In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
# add the repo root (the folder containing config.py) to the path, from any depth
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "config.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import config, pandas as pd
from cprics.io_data import load_peak_amplitudes
from cprics.islands import build_all_islands

pk  = load_peak_amplitudes(config.PEAK_AMP_DIR, "9_top_right")
isl = build_all_islands(pk, wavelength=9)       # gap = 1.5 x 9 = 13.5 m

print("islands found:", len(isl))
print("sub-threshold (<18 mm):", (isl.amplitude < 18).sum(),
      f"({(isl.amplitude < 18).mean()*100:.0f}%)")
print("exceedances   (>=18 mm):", (isl.amplitude >= 18).sum())
print("\naccumulated amplitude (mm):\n", isl.amplitude.describe().round(2).to_string())
print("\npeaks per island:\n", isl.n_peaks.describe().round(2).to_string())

islands found: 1152
sub-threshold (<18 mm): 702 (61%)
exceedances   (>=18 mm): 450

accumulated amplitude (mm):
 count    1152.00
mean       19.05
std         7.47
min        12.15
25%        14.70
50%        16.73
75%        20.56
max        63.58

peaks per island:
 count    1152.00
mean        3.55
std         1.12
min         3.00
25%         3.00
50%         3.00
75%         4.00
max        10.00


In [2]:
# rough sanity check against the incumbent (9 m, right rail)
from cprics.cyc import load_all_cyc
cyc9 = load_all_cyc(config.CYC_DIR).query("wavelength == '9' and rail == 'R'")
ours = isl[isl.amplitude >= 18]
print("our >=18 islands:", len(ours), "| CYC 9m/R exceedances:", len(cyc9))
print("amplitude median — ours:", round(ours.amplitude.median(), 1),
      "| CYC:", round(cyc9.amplitude.median(), 1))
print("our amplitude-per-peak median:", round((ours.amplitude / ours.n_peaks).median(), 2),
      "(expect ~5-6 if the accumulation matches)")

our >=18 islands: 450 | CYC 9m/R exceedances: 284
amplitude median — ours: 22.0 | CYC: 21.7
our amplitude-per-peak median: 5.7 (expect ~5-6 if the accumulation matches)
